<center style="padding:1rem 0;">
    <h1 style="font-size: 4rem;">IA - Deep Learning</h1>
    <h2 style="font-size: 2rem;">Livrable 2 - Construction d'un premier réseau de neurones</h2>
    <h5 style="font-size: 1rem;"><i>Thomas VINET, Hugo HELM, Alban GODIER</i></h5>
</center>

<img src="assets/cesi.png" style="position:absolute;right:2rem;top:4.5rem;width:10rem;background:#fee237;"/>

In [1]:
from __future__ import annotations
from typing import Optional, Dict, Tuple, List

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.neural_network import NeuralNetwork, DrawRealTimeLoss, EarlyStopping, Layer, Evaluation
from lib.neural_network.grid_search import GridSearch

warnings.filterwarnings("ignore")
np.random.seed(42)

## Démonstration mathématiques

### Démonstration de la descente de gradient

Lors de la descente de gradient dans note modèle, nous cherchons à ajuster les poids et les biais de chaque couche du réseau de neurone. Pour ce faire, nous partons de la sortie du réseau pour chercher à optimiser le résultat de la fonction de coût. 

#### Introduction des variables

On considère un réseau de neurones entièrement connecté, composé de couches indexées par $i \in \{0, \dots, L\}$ où :
- $i = 0$ : couche de sortie
- $i = L$ : couche d’entrée

On pose :
- $y$ : la valeur attendue pour la prédiction
- $\hat y$ : la prédiction du modèle
- $\mathcal L$ la fonction de coût du modèle, définie en fonction de $y$ et $\hat y$
- $A_i$ la fonction d'activation de la couche $i$, appliquée composant par composant
- $z_{i}$ la matrice des sorties des fonctions d'agrégation des neurones de la couche $i$
- $x_i$ la matrice des entrées pour les neurones de la couche $i$
- $w_{i}$ la matrice des poids liés aux entrées $x_i$ pour les neurone de la couche $i$
- $b_{i}$ la matrice des biais des neurones de la couche $i$

Les dimensions des matrices sont les suivantes :
- $x_L$ : $\text{variables entrée} \times 1$ , la matrice a une ligne par variable d'entrée (sortie de la couche $i+1$) et 1 colonne.
- $x_i$ : $\text{Neurones}_{i+1} \times 1$, la matrice a une ligne par neurone de la couche précédente ($i+1$) et 1 colonne.
- $w_i$ : $\text{Neurones}_i \times \text{Neurones}_{i+1}$ , la matrice a une ligne par neurone de la couche $i$ et une colonne par neurone de la couche $i+1$
- $b_i$ : $\text{Neurones}_i \times 1$ , la matrice a une ligne par neurone de la couche $i$ et 1 colonne.
- $z_i$ : $\text{Neurones}_i \times 1$, la matrice a une ligne par neurone de la couche $i$ et 1 colonne.

Pour chaque couche du réseau :
$$
\begin{cases}
z_i = w_ix_i + b_i \\
x_{i-1} = A_i(z_i)
\end{cases}
$$

On note également la prédiction du modèle et la fonction de coût :
- $\hat y = A_0(z_0)$
- $\mathcal L = \mathcal L(y, \hat y)$

Dans le cas de la descente de gradient, on cherche à connaitre :
$$\frac{\partial \mathcal L}{\partial w_i} \quad et \quad \frac{\partial \mathcal L}{\partial b_i}$$
Pour cela, on pose la variable :
$$\delta_i = \frac{\partial \mathcal L}{\partial z_i}$$

Nous cherchons donc à définir tous les gradients des poids et des biais en fonction de ces valeurs.
Afin d'avoir une formule plus dynamique pour le modèle, nous voulons obtenir des expressions récurrentes, en afin d'obtenir le gradient d'une couche $i$ en fonction de la couche précédente ($i-1$)

#### Exemple pour la couche 0

Afin de démontrer notre relation récursive, on commence par définir le gradient de la dernière couche (couche de sortie).
On a :
- $\hat y = A_0(z_0)$, $A_0$ étant la fonction d'activation de la dernière couche
- $z_0=w_0x_0+b_0$

La matrice $\hat y$ est de forme $\text{Neurones}_0 \times 1$ (matrice colonne dont le nombre de ligne correspond au nombre de sortie de la couche de sortie du modèle)

On cherche : $\frac{\partial \mathcal L}{\partial w_0}$ et $\frac{\partial \mathcal L}{\partial b_0}$
Pour cela on cherche $\delta_0$ afin de les obtenir par composition.


On pose :
$$
\begin{align}
\delta_0 &= \frac{\partial \mathcal L}{\partial \hat y} \odot \frac{\partial \hat y}{\partial z_0} \\
&= \frac{\partial \mathcal L}{\partial \hat y} \odot A'_0(z_0)
\end{align}
$$
Donc :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial w_0}
	& =
	\delta_0 \cdot \left ( \frac{\partial z_0}{\partial w_0} \right )^\intercal \\
	& =
	\delta_0 \cdot x_0^\intercal
\end{align}
$$
Comme $\delta_0$ est une matrice colonne ($\text{Neurones}_0 \times 1$) et $x_0$ ($\text{Neurones}_1 \times 1$) l'est aussi, on multiplie $\delta_0$ par par la transposée de $x_0$. de cette manière, on obtient une matrice de forme $\text{Neurones}_0 \times \text{Neurones}_1$ que l'on peut donc soustraire à $w_0$. 
Et :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial b_0}
	& =
	\delta_0 \cdot \frac{\partial z_0}{\partial b_0} \\
	& =
	\delta_0
\end{align}
$$

#### Exemple pour la couche 1

On définit ensuite le gradient de notre première couche cachée (couche 1).
On a :
- $z_0=w_0 x_0 + b_0$
- $x_0=A_1(z_1)$
- $z_1=w_1 x_1 + b_1$

On cherche : $\frac{\partial \mathcal L}{\partial w_1}$ et $\frac{\partial \mathcal L}{\partial b_1}$
Pour cela on cherche $\delta_1$, comme pour la dernière couche.
On pose :
$$
\begin{align}
	\delta_1
	& =
	\left ( \left ( 
	 \frac
	  {\partial z_0}
	  {\partial x_0}
	 \right )^\intercal
	\cdot
	\frac
	 {\partial \mathcal L}
	 {\partial \hat y} 
	\cdot 
	\frac
	 {\partial \hat y}
	 {\partial z_0}
	\right )
	\odot
	\frac
	 {\partial x_0}
	 {\partial z_1} \\
	& =
	(	w_0^\intercal
	\cdot
	\delta_0)
	\odot
	A_1'(z_1)
\end{align}
$$
$w_0$ est de dimensions $\text{Neurones}_0 \times \text{Neurones}_1$ et $\delta_0$ est de dimensions $\text{Neurones}_0 \times 1$ nous calculons la transposée de $w_0$ avant de la multiplier avec $\delta_0$.
On obtient alors une matrice de dimensions $\text{Neurones}_1 \times 1$ qui peut être multipliée par élément avec $A_1'(z_1)$ qui est aussi de dimensions $\text{Neurones}_1 \times 1$.
Donc :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial w_1}
	& =
	\delta_1 \cdot \left ( \frac{\partial z_1}{\partial w_1} \right )^\intercal \\
	& =
	\delta_1 \cdot x_1^\intercal
\end{align}
$$
Et :
$$
\begin{align}
	\frac{\partial \mathcal L}{\partial b_1}
	& =
	\delta_1 \cdot \frac{\partial z_1}{\partial b_1}  \\
	& =
	\delta_1
\end{align}
$$

#### Définition de la relation récursive

En partant de l'expression de la couche 1 :
$$
\begin{align}
	\delta_1
	& =
	(
	w_0^\intercal
	\cdot
	\delta_0)
	\odot
	A_1'(z_1)
\end{align}
$$
On a les dimensions :
- $\delta_i$ : $\text{Neurones}_i \times 1$
- $A_i'(z_i)$ : $\text{Neurones}_i \times 1$

En prenant se basant sur les exemples précédents, on peut définir la relation suivante :

$$
\begin{align}
	\delta_i
	& =
	(w_{i-1}^\intercal
	\cdot
	\delta_{i-1})
	\odot
	A_i'(z_i)
\end{align}
$$
Pour les poids :
$$
\frac{\partial \mathcal L}{\partial w_i}
=
\delta_i \times x_i^\intercal
$$
Qui est donc de dimensions : $\text{Neurones}_i \times \text{Neurones}_{i+1}$

Pour les biais :
$$
\frac{\partial \mathcal L}{\partial b_i} = \delta_i
$$
Qui est donc de dimensions : $\text{Neurones}_i \times 1$

**Cas des batches**

Dans le cas des batches, les dimensions de certaines matrices changent :
- $x_L$ : $\text{variables entrée} \times \text{batches}$ , la matrice a une ligne par variable d'entrée et une colonne par batch.
- $x_i$ : $\text{Neurones}_{i+1} \times \text{batches}$, la matrice a une ligne par neurone de la couche précédente ($i+1$) et une colonne par batch.
- $z_i$ : $\text{Neurones}_i \times \text{batches}$, la matrice a une ligne par neurone de la couche $i$ et une colonne par batch.

Cela a principalement un effet sur les dimensions de la sortie :
- $\hat y$ : $1 \times \text{batches}$

Ce qui change donc la formule du gradient des biais, comme $\delta_i$ est désormais de dimension $\text{Neurones}_i \times \text{batches}$ (dépendant de $\hat y$), cette formule devient donc :
$$\frac{\partial \mathcal L}{\partial b_i} = \sum_{\text{batches}}(\delta_i)$$
Où $\sum_{\text{batches}}(\delta_i)$ est la somme de $\delta_i$ calculée sur chaque ligne, ce qui renvoie une matrice de dimension $\text{Neurones}_i \times 1$.

Comme la moyenne est déjà faite sur la MSE, il n'y a pas besoin de la faire pour les gradients des biais et des poids.

### Dérivées des fonctions de coûts et d'activation

#### Fonction d'activation 

**Sigmoïde :**
$$
    \sigma(x) 
    =
    \frac{1}{1+e^{-x}}
$$
$$
    \sigma'(x)
    =
    \frac{e^{-x}}{(1+e^{-x})^2}
    =
    \sigma(x)(1-\sigma(x))
$$

**ReLU :**
$$
    a(x)=\max(0,x)
$$
$$
    a'(x)=
    \begin{cases}
     \begin{align}
      & 0 & \text{pour}\ x<0 \\
      & 1 & \text{pour}\ x\ge 0
     \end{align}
    \end{cases}
$$

**Tanh :**
$$
    a(x)
    =
    \tanh(x)
$$
$$
    a'(x)
    =
    \frac
     {1}
     {\cosh(x)^2}
    =
    1-\tanh(x)^2
$$

#### Fonction de coût

**Cross-entropy :**
$$
    \mathcal L
    =
    -y \log(\hat y) - (1- y) \log(1- \hat y)
$$
$$
    \frac{\partial \mathcal L}{\partial \hat y}
    =
    -\frac{y}{\hat y} - \frac{1-y}{1 - \hat y}
$$

**Mean squared error (MSE) :**
$$
    MSE 
    = 
    \frac{1}{n}
    \sum^n_{j=1}{(y_j - \hat y_j)^2}
$$
$$
    \frac{\partial \mathcal L}{\partial \hat y_j}
    =
    \frac{2}{n} (\hat y_j - y_j)
$$

**Mean absolute error (MAE) :**
$$
    MAE 
    = 
    \frac{1}{n}
    \sum^n_{j=1}{|y_j - \hat y_j|}
$$
$$
    \frac{\partial\mathcal L}{\partial \hat y_i}
    =
    \begin{cases}
     \begin{align}
      & 1 & \text{pour}\ \hat y_i > y_i \\ 
      & -1 & \text{pour}\ \hat y_i < y_i\\
      & \text{indéfini} & \text{pour}\ \hat y_j = y_j
      \end{align}
    \end{cases}
$$

## 1. Chargement des données

Chargement des données d'entraînement et de validation à partir des fichiers CSV.

In [2]:
# Chargement des données
df_train = pd.read_csv('dataset/dataset_train.csv')
df_validation = pd.read_csv('dataset/dataset_validation.csv')

# Pour tester, on peut réduire la taille des données d'entraînement
# df_train = df_train.sample(n=10000, random_state=42).reset_index(drop=True)
# df_validation = df_validation.sample(n=2000, random_state=42).reset_index(drop=True)

print(f"Données d'entraînement: {df_train.shape}")
print(f"Données de validation: {df_validation.shape}")
print(f"\nPremières lignes des données d'entraînement:")
print(df_train.head())
print(f"\nInformations sur les données:")
print(df_train.info())

Données d'entraînement: (59582, 17)
Données de validation: (22384, 17)

Premières lignes des données d'entraînement:
   Diabetes_binary  HighBP  HighChol  CholCheck      BMI  Smoker  Stroke  \
0              0.0     0.0       0.0        1.0  0.31250     0.0     0.0   
1              0.0     1.0       0.0        1.0  0.34375     1.0     0.0   
2              1.0     1.0       1.0        1.0  0.75000     0.0     0.0   
3              1.0     1.0       0.0        1.0  0.53125     0.0     0.0   
4              0.0     1.0       1.0        1.0  0.43750     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  HvyAlcoholConsump  \
0                   0.0           1.0     1.0      1.0                0.0   
1                   0.0           1.0     0.0      1.0                0.0   
2                   1.0           0.0     1.0      1.0                0.0   
3                   0.0           0.0     0.0      1.0                0.0   
4                   0.0           0.0    

In [3]:
# Séparation des features et de la cible
# La colonne cible est 'Diabetes_binary'
target_column = 'Diabetes_binary'

X_train = df_train.drop(columns=[target_column]).astype(float)
y_train = df_train[target_column].astype(int)

X_validation = df_validation.drop(columns=[target_column]).astype(float)
y_validation = df_validation[target_column].astype(int)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_validation shape: {X_validation.shape}, y_validation shape: {y_validation.shape}")
print(f"\nDistribution de y_train: {np.bincount(y_train)}")
print(f"Distribution de y_validation: {np.bincount(y_validation)}")

X_train shape: (59582, 16), y_train shape: (59582,)
X_validation shape: (22384, 16), y_validation shape: (22384,)

Distribution de y_train: [29791 29791]
Distribution de y_validation: [19074  3310]


In [ ]:
# Définition des architectures et paramètres pour la grid search
from lib.neural_network.activation.relu import Relu
from lib.neural_network.activation.sigmoid import Sigmoid
from lib.neural_network.activation.none import NoActivation
from lib.neural_network.loss.mean_squared_error import MeanSquaredError
from lib.neural_network.loss.binary_cross_entropy import BinaryCrossEntropy
from lib.neural_network.loss.binary_cross_entropy_sigmoid import BinaryCrossEntropySigmoid
from lib.neural_network.grid_search import GridSearch, Params
from lib.neural_network.callback.draw_real_time_loss import DrawRealTimeLoss
from lib.neural_network.callback.progress_bar import ProgressBar

nn = NeuralNetwork([
    Layer(neurons=64, activation=Relu()),
    Layer(neurons=32, activation=Relu()),
    Layer(neurons=1, activation=Sigmoid()),
], loss=MeanSquaredError(), inputs=X_train.shape[1])

nn.add_callback(EarlyStopping())
# nn.add_callback(DrawRealTimeLoss())
# nn.add_callback(ProgressBar())

nn.fit(
    X_train.to_numpy(),
    y_train.to_numpy(),
    epochs=100,
    batch_size=32,
    learning_rate=0.01
)

evaluation = Evaluation(
    X_validation.to_numpy(),
    y_validation.to_numpy()
)
print(evaluation.validate(nn))
evaluation.draw_loss_history()
evaluation.draw_roc()

Epoch 1/100
1490/1490 ━━━━━━━━━━━━━━━━━━━━ 923ms 1ms/step - loss: 0.0000 - val_loss: 0.0000
Epoch 1/100 - 951ms - loss: 0.1826 - val_loss: 0.1903

Epoch 2/100
1490/1490 ━━━━━━━━━━━━━━━━━━━━ 1.0s 1ms/step - loss: 0.1826 - val_loss: 0.190303
Epoch 2/100 - 1.0s - loss: 0.1759 - val_loss: 0.1894

Epoch 3/100
1260/1490 ━━━━━━━━━━━━━━━━     767ms 1ms/step - loss: 0.1759 - val_loss: 0.1894

## 2. Choix de l'Architecture du Réseau

### Variation de l'architecture avec différentes configurations
- Couches et nombre de neurones
- Fonctions d'activation : Sigmoid, ReLU, Tanh, Softmax
- Fonctions de perte : MSE, MAE, BCE, CCE
- Dropouts pour la régularisation

In [ ]:
# Définition des architectures et paramètres pour la grid search
from lib.neural_network.activation.relu import Relu
from lib.neural_network.activation.sigmoid import Sigmoid
from lib.neural_network.loss.mean_squared_error import MeanSquaredError
from lib.neural_network.loss.binary_cross_entropy import BinaryCrossEntropy
from lib.neural_network.grid_search import GridSearch, Params

# Paramètres pour la grid search
grid_search_params: Params = {
    "learning_rate": np.linspace(0.01, 0.001, 5),
    "batch_size": [32],
    "epochs": [100],
    "loss": [MeanSquaredError(), BinaryCrossEntropy()],
    "early_stopping_patience": [5],
    "architecture": [
        # Architecture simple : 1 couche cachée
        # [
        #     {"neurons": [32], "dropout_rate": [0.2], "activation": [Relu()]},
        #     {"neurons": [1], "dropout_rate": [0.0], "activation": [Sigmoid()]},
        # ],
        # Architecture moyenne : 2 couches cachées
        [
            {"neurons": [64], "dropout_rate": [0.2], "activation": [Relu()]},
            {"neurons": [32], "dropout_rate": [0.2], "activation": [Relu()]},
            {"neurons": [1], "dropout_rate": [0.0], "activation": [Sigmoid()]},
        ],
        # [
        #     {"neurons": [64], "dropout_rate": [0.2], "activation": [Sigmoid()]},
        #     {"neurons": [32], "dropout_rate": [0.2], "activation": [Sigmoid()]},
        #     {"neurons": [1], "dropout_rate": [0.0], "activation": [Sigmoid()]},
        # ],
        # Architecture profonde : 3 couches cachées
        # [
        #     {"neurons": [128], "dropout_rate": [0.3], "activation": [Relu()]},
        #     {"neurons": [64], "dropout_rate": [0.2], "activation": [Relu()]},
        #     {"neurons": [32], "dropout_rate": [0.2], "activation": [Relu()]},
        #     {"neurons": [1], "dropout_rate": [0.0], "activation": [Sigmoid()]},
        # ],
    ],
}

print(
    f"Nombre total de combinaisons : {len(grid_search_params['learning_rate']) * len(grid_search_params['batch_size']) * len(grid_search_params['epochs']) * len(grid_search_params['loss']) * len(grid_search_params['early_stopping_patience']) * len(grid_search_params['architecture'])}"
)

### 2.1 Comparaison des architectures

Entraînement et comparaison de plusieurs architectures pour déterminer la meilleure configuration.

In [ ]:
# Exécution de la grid search avec comparaison des architectures
gs = GridSearch(num_threads=12)

print("Démarrage de la recherche en grille (Grid Search)...")
print("="*80)

result = gs.search_and_compare(
    grid_search_params,
    X_train.to_numpy(),
    y_train.to_numpy(),
    X_validation.to_numpy(),
    y_validation.to_numpy(),
)

print("="*80)
print("Grid Search terminée !")


In [ ]:
nn.predict(X_validation.to_numpy()).sum() / X_validation.shape[1]

## 3. Construction et Entraînement du Réseau Final

### Utilisation de la meilleure architecture avec Early Stopping et callbacks

## 4. Entraînement, Évaluation et Métriques

### Liste des métriques de performance

#### Accurcy

L'accuracy est la proportion de prédictions correctes par rapport au nombre total de prédictions. Elle est calculée de la manière suivante :
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
Où :
- $TP$ (True Positives) : nombre de vrais positifs
- $TN$ (True Negatives) : nombre de vrais négatifs
- $FP$ (False Positives) : nombre de faux positifs
- $FN$ (False Negatives) : nombre de faux négatifs

Cette métrique indique la proportion de prédictions correctes, mais elle peut être trompeuse dans le cas de classes déséquilibrées. Par exemple, si une classe représente 95% des données, un modèle qui prédit toujours cette classe aura une accuracy de 95%, même s'il ne fait aucune prédiction correcte pour l'autre classe.

#### Precision

La précision (precision) est la proportion de prédictions positives correctes par rapport au nombre total de prédictions positives. Elle est calculée de la manière suivante :
$$\text{Precision} = \frac{TP}{TP + FP}$$

Cette métrique mesure la capacité du modèle à ne pas faire de fausses prédictions positives. Une précision élevée signifie que lorsque le modèle prédit une classe positive, il a de fortes chances d'être correct.

#### Recall

Le rappel (recall) est la proportion de prédictions positives correctes par rapport au nombre total de cas positifs réels. Il est calculé de la manière suivante :
$$\text{Recall} = \frac{TP}{TP + FN}$$

Cette métrique mesure la capacité du modèle à identifier tous les cas positifs. Un rappel élevé signifie que le modèle a de fortes chances de détecter les cas positifs, même s'il fait quelques erreurs de classification.

#### F1-Score

Le F1-Score est la moyenne harmonique de la précision et du rappel. Il est calculé de la manière suivante :
$$\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Cette métrique combine à la fois la précision et le rappel en une seule valeur, ce qui est utile lorsque les classes sont déséquilibrées. Un F1-Score élevé indique que le modèle a à la fois une bonne précision et un bon rappel.

#### AUC-ROC

L'AUC-ROC (Area Under the Receiver Operating Characteristic Curve) mesure la capacité du modèle à distinguer entre les classes positives et négatives. Elle est calculée en traçant la courbe ROC, qui représente le taux de vrais positifs (TPR) en fonction du taux de faux positifs (FPR) à différents seuils de classification. L'AUC-ROC varie entre 0 et 1, où une valeur de 1 indique une parfaite séparation des classes, tandis qu'une valeur de 0.5 indique une performance équivalente à un classificateur aléatoire.

### Métriques d'évaluation du modèle

Dans notre cas, les métriques les plus importantes dans l'évaluation de notre modèle sont les suivantes, dans cet ordre de priorité :
- Recall : Nous cherchons à identifier en priorité les cas de diabète positifs, quitte à faire des erreurs sur les cas négatifs (il est préférable de faire des faux positifs que des faux négatifs dans ce cas, car les faux négatifs peuvent avoir des conséquences graves pour les patients).
- Precision : Nous voulons également, de manière secondaire, limiter les faux positifs afin d'éviter de faire des test inutiles pour les patients qui n'ont pas de diabète.
- F1-Score : En tant que métrique combinant à la fois la précision et le rappel, le F1-Score nous permettra d'avoir une évaluation globale de la performance du modèle. 

## 5 Détection du Surapprentissage (Overfitting)

Analyse des courbes d'entraînement pour détecter le surapprentissage

## 6. Analyse du Seuil de Décision

Variation du seuil de décision pour trouver la meilleure configuration

## 7. Résumé et Conclusions

### Synthèse des résultats et recommandations

### 6.1 Tableau comparatif - Résultats des architectures testées